<a href="https://colab.research.google.com/github/YOUR-USERNAME/bags-vectors-transformers/blob/main/day2/notebooks/2_bonus_embeddings_classifier_exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bags, Vectors & Transformers
## Day 2 — Embeddings as Features for Classification *(Bonus)*

**A Methods Workshop in Computational Text Analysis**
Denise J. Roth · Strategic Communication Group · Wageningen University & Research

---

On **Day 1** we trained a supervised classifier on **TF-IDF** features. We have since learned
about **embeddings**. This bonus notebook brings the two together on a real **policy** task:
classifying **US Congressional bills** by their **policy area** (Health, Education, Taxation,
and so on), from the bill **title**.

The data is the **`dreamproit/bill_labels_us`** dataset — ~119,000 bills whose policy areas
were assigned by expert analysts at the **Congressional Research Service** of the Library of
Congress. It is public-domain, so it loads with no account or key.

By the end you will be able to:

- Turn a document into a single vector by **averaging** its word embeddings
- Train a classifier on those embedding features
- **Compare** embeddings vs. a TF-IDF baseline on identical data
- Reason about *when* embeddings help and when they don't

> The honest lesson: embeddings are a powerful **option**, not a guaranteed upgrade. We let
> the held-out data decide — the same discipline as Day 1.


## 0. Setup

In [ ]:
!pip install datasets gensim -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

import gensim.downloader as api

print("Setup complete!")

## 1. Loading the policy data

We load the bills dataset and keep just what we need: the bill **title** (short text) and its
**policy area** (the label). To keep the workshop fast, we focus on a handful of common policy
areas and take a manageable sample.


In [ ]:
from datasets import load_dataset

# Load the training split (public domain, no key needed).
bills = load_dataset("dreamproit/bill_labels_us", split="train")
df = bills.to_pandas()

print("Total bills:", len(df))
print("Columns:", list(df.columns))
df[["title", "policy_area"]].head()

The full dataset has many policy areas, some with very few bills. We keep the
**most common** areas and balance the sample so no single class dominates.


In [ ]:
# Use the bill title as our text, and policy_area as the label
df = df.rename(columns={"title": "text"})
df = df[["text", "policy_area"]].dropna()

# Keep the 6 most frequent policy areas for a clean multi-class demo
top_areas = df["policy_area"].value_counts().head(6).index.tolist()
df = df[df["policy_area"].isin(top_areas)]

# Balance: sample up to 800 bills per area, then shuffle
df = (df.groupby("policy_area", group_keys=False)
        .apply(lambda g: g.sample(min(len(g), 800), random_state=42))
        .sample(frac=1, random_state=42)
        .reset_index(drop=True))

print("Policy areas kept:")
print(df["policy_area"].value_counts())

Let's look at a few examples so we know what the model is working with.


In [ ]:
for area in df["policy_area"].unique()[:5]:
    example = df[df["policy_area"] == area]["text"].iloc[0]
    print(f"[{area}]")
    print(f"   {example}\n")

Now we split into train and test, and load pre-trained **GloVe** vectors.


In [ ]:
train_df, test_df = train_test_split(
    df, test_size=0.25, random_state=42, stratify=df["policy_area"]
)
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)
print("Train:", len(train_df), "| Test:", len(test_df))

In [ ]:
# Load pre-trained GloVe (50-dim, small & fast). ~66 MB on first run.
glove = api.load("glove-wiki-gigaword-50")
print("GloVe loaded. Vocab size:", len(glove), "| dim:", glove.vector_size)

## 2. A light clean

Bill titles are fairly clean already, so we just lowercase and keep alphabetic tokens.


In [ ]:
import re

def clean(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    return text.split()

train_df["tokens"] = train_df["text"].apply(clean)
test_df["tokens"] = test_df["text"].apply(clean)

y_train = train_df["policy_area"].values
y_test = test_df["policy_area"].values

print("Example tokens:", train_df["tokens"].iloc[0][:12])

## 3. Baseline: TF-IDF features (the Day 1 approach)

First we reproduce the Day 1 pipeline as our **baseline** to beat.


In [ ]:
train_text = train_df["tokens"].apply(" ".join)
test_text = test_df["tokens"].apply(" ".join)

tfidf = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf.fit_transform(train_text)
X_test_tfidf = tfidf.transform(test_text)

clf_tfidf = LogisticRegression(max_iter=1000)
clf_tfidf.fit(X_train_tfidf, y_train)
pred_tfidf = clf_tfidf.predict(X_test_tfidf)

acc_tfidf = accuracy_score(y_test, pred_tfidf)
print(f"TF-IDF baseline accuracy: {acc_tfidf:.3f}")

## 4. Embedding features: averaging word vectors

Now the new part. We turn each bill title into a single vector by **averaging** the GloVe
vectors of its words (skipping words not in GloVe's vocabulary). This is the words-to-document
bridge from the lecture.


In [ ]:
def document_vector(tokens, model=glove):
    """Average the embedding vectors of the tokens in a document."""
    vectors = [model[t] for t in tokens if t in model]
    if len(vectors) == 0:
        return np.zeros(model.vector_size)
    return np.mean(vectors, axis=0)

X_train_emb = np.vstack([document_vector(toks) for toks in train_df["tokens"]])
X_test_emb = np.vstack([document_vector(toks) for toks in test_df["tokens"]])

print("Embedding feature matrix shape:", X_train_emb.shape)
print("(That's", X_train_emb.shape[0], "bills x", X_train_emb.shape[1], "dimensions.)")

Notice the shape: just **50 columns**, versus thousands for TF-IDF. Embedding features
are **dense and compact**. Now train the *same* classifier on them.


In [ ]:
clf_emb = LogisticRegression(max_iter=1000)
clf_emb.fit(X_train_emb, y_train)
pred_emb = clf_emb.predict(X_test_emb)

acc_emb = accuracy_score(y_test, pred_emb)
print(f"Embedding-feature accuracy: {acc_emb:.3f}")

## 5. Head-to-head comparison

The moment of truth: which representation won on this policy-classification task?


In [ ]:
print(f"TF-IDF baseline:      {acc_tfidf:.3f}")
print(f"Embedding features:   {acc_emb:.3f}")
print()
diff = acc_emb - acc_tfidf
if diff > 0:
    print(f"Embeddings won by {diff:.3f}")
elif diff < 0:
    print(f"TF-IDF won by {-diff:.3f}")
else:
    print("A tie!")

In [ ]:
plt.figure(figsize=(6, 5))
plt.bar(["TF-IDF\n(baseline)", "Embeddings\n(averaged)"],
        [acc_tfidf, acc_emb],
        color=["#1A1A2E", "#34B233"])
plt.ylabel("Test accuracy")
plt.title("TF-IDF vs. embedding features (same classifier, same data)")
plt.ylim(0, 1)
for i, v in enumerate([acc_tfidf, acc_emb]):
    plt.text(i, v + 0.02, f"{v:.3f}", ha="center", fontweight="bold")
plt.tight_layout()
plt.show()

**Don't be surprised if TF-IDF holds its own — or even wins.** Bill titles contain
strong keyword signals ("tax", "health", "school"), which is exactly what TF-IDF thrives on.
Averaging embeddings blurs those signals together. Embeddings tend to pull ahead when meaning
and generalization matter more than exact keywords.

> **✏️ Exercise 1**
>
> Which representation won, and by how much? Write a sentence interpreting the result. Given
> that these are short, keyword-heavy bill titles, does the outcome make sense?


In [ ]:
# Your answer here (as a comment or markdown)


## 6. Improving the embedding features: TF-IDF weighting

A plain average treats every word equally. We can do better: **weight each word's vector by
its TF-IDF score** before averaging, so distinctive words (like "immigration" or "medicare")
count more than common ones (like "act" or "bill").


In [ ]:
idf = dict(zip(tfidf.get_feature_names_out(), tfidf.idf_))

def weighted_document_vector(tokens, model=glove, idf=idf):
    """TF-IDF-weighted average of word vectors."""
    vectors, weights = [], []
    for t in tokens:
        if t in model and t in idf:
            vectors.append(model[t])
            weights.append(idf[t])
    if not vectors:
        return np.zeros(model.vector_size)
    vectors = np.array(vectors)
    weights = np.array(weights).reshape(-1, 1)
    return (vectors * weights).sum(axis=0) / weights.sum()

X_train_w = np.vstack([weighted_document_vector(toks) for toks in train_df["tokens"]])
X_test_w = np.vstack([weighted_document_vector(toks) for toks in test_df["tokens"]])

clf_w = LogisticRegression(max_iter=1000)
clf_w.fit(X_train_w, y_train)
pred_w = clf_w.predict(X_test_w)
acc_w = accuracy_score(y_test, pred_w)

print(f"Plain average:            {acc_emb:.3f}")
print(f"TF-IDF weighted average:  {acc_w:.3f}")

> **✏️ Exercise 2**
>
> Did TF-IDF weighting help? Add its bar to the comparison chart from Section 5 (three bars:
> TF-IDF, plain embeddings, weighted embeddings). Which is best on this task?


In [ ]:
# Your code here


## 7. Why might embeddings help? A qualitative look

Even when accuracy is similar, embeddings capture something TF-IDF cannot: **meaning beyond
exact words**. Two bill titles about the same policy area can use *different* vocabulary.


In [ ]:
def cosine(a, b):
    if np.all(a == 0) or np.all(b == 0):
        return 0.0
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

t1 = "a bill to lower prescription drug prices".split()
t2 = "an act to reduce the cost of medicine".split()     # same topic, different words
t3 = "a bill to modernize the naval fleet".split()        # different topic

v1, v2, v3 = document_vector(t1), document_vector(t2), document_vector(t3)

print("Same policy area, different words (health):")
print(f"  t1 vs t2: {cosine(v1, v2):.3f}")
print("Different policy area (defense):")
print(f"  t1 vs t3: {cosine(v1, v3):.3f}")

> **✏️ Exercise 3**
>
> Compare that to what **TF-IDF** would see. `t1` and `t2` share almost no content words, so
> their TF-IDF vectors would be nearly orthogonal (similarity ≈ 0). Confirm this by
> transforming the three titles with the fitted `tfidf` vectorizer and computing cosine
> similarities. What does this tell you about the *kind* of task where embeddings should win?


In [ ]:
# Your code here


## Wrap-up

You have now used embeddings as **classification features** on a real policy task and compared
them head-to-head with TF-IDF on identical data:

- Classified **Congressional bills by policy area** from their titles
- Turned titles into vectors by **averaging** word embeddings (plain and TF-IDF weighted)
- Trained the **same classifier** on both representations
- Found — as the lecture warned — that embeddings are **not automatically better**, especially
  on short, keyword-rich text
- Saw *why* they can help: they capture **meaning beyond exact word overlap**

The practical takeaway matches Day 1's discipline: **try both, and let the held-out data
decide.** There is no representation that is best for every task.

### Optional challenge

1. **Concatenate** TF-IDF and embedding features into one matrix and train on the combination
   — sometimes the best of both. *(Hint: `scipy.sparse.hstack`, or densify the TF-IDF.)*
2. Look at the **confusion matrix** for the embedding classifier (from Day 1's code): which
   policy areas get confused with each other? Do the confusions make intuitive sense?


In [ ]:
# Optional challenge — your code here
